# v124c — cons2of3_c26 fold-2 honest 複製 eval (Colab、条件付き)

v120 の fold-2 +0.01155 は crop det の漏洩で撤回済み (§2.98 開示)。
**v121c (fold-0) ゲート通過時のみ実行**する第二 fold 確認 — crop det を
v124a honest 版 (folds(0,1) のみ学習) に差し替え、v120 と同一の 6 arm・
評価経路で fold-2 OOF を再測定:

| arm | 放出ルール |
|---|---|
| `prod` | yolo 提案のみ conf>=0.26 (sanity ≈0.444238) |
| `union3` | 全クラスタ (union 参照) |
| `union3_c26` | max conf>=0.26 のクラスタ |
| `cons2of3` | >=2 detector 支持 (低conf合意救済) |
| `cons2of3_c26` | conf>=0.26 の支持 >=2 (複製対象 arm) |
| `cons3of3` | 全員支持 |

**判定** (§2.99 補完):
- cons2of3_c26 Δ@0.45 ≥ +0.005 → `second_fold_positive`
- 未満 → `second_fold_negative` (fold-0 陽性は未再現)
- sanity: prod arm ≈ v76 実績 0.444238 ±0.001
- 片 fold 陽性だけでは提出差替えの根拠にしない

**Provenance pin:** `V73_PROVENANCE_SHA256` (v73-v120 と同一 fold-2
manifest)。yolo det は manifest binding、crop/yolo26 det は各学習
manifest 記録の weights_sha256 を trusted-pin として受理。
refiner は prod baseline のみ (manifest role=baseline)。
Drive `solar_filament_v35/kaggle.json` で kaggle 認証 (shingo257)。
eval 終了時に Drive `solar_filament_v124/work/latest/` へ sha256 検証付きミラー。


In [ ]:
import os, re, json, subprocess, sys, hashlib, shutil, time
from pathlib import Path
# Reviewed provenance pin is supplied externally — env var, Colab secret,
# Kaggle secret — or declared interactively. Never derived from artifacts.
EXPECTED_PROVENANCE_SHA = os.environ.get('V73_PROVENANCE_SHA256', '')
if not EXPECTED_PROVENANCE_SHA:
    try:
        from google.colab import userdata
        EXPECTED_PROVENANCE_SHA = userdata.get('V73_PROVENANCE_SHA256') or ''
    except Exception:
        pass
if not EXPECTED_PROVENANCE_SHA:
    try:
        from kaggle_secrets import UserSecretsClient
        EXPECTED_PROVENANCE_SHA = UserSecretsClient().get_secret('V73_PROVENANCE_SHA256')
    except Exception:
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    try:
        EXPECTED_PROVENANCE_SHA = input('V73_PROVENANCE_SHA256 (reviewed manifest hash): ').strip()
    except (EOFError, OSError):
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    raise RuntimeError('Set V73_PROVENANCE_SHA256 after independent provenance review')

# [0] Drive mount — primary data source (same pattern as all past colab runs)
from google.colab import drive
drive.mount('/content/drive')
V30 = Path('/content/drive/MyDrive/solar_filament_v30')
COMP = V30 / 'competition_data/MAGFiLO_1.0_Kaggle_2026'
inputs = {
    'train_json': COMP / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    'test_dir': COMP / 'test/test_images',
    'preproc_dir': V30 / 'preprocessed_v30_q96',
    'folds_csv': V30 / 'artifacts/grouped_folds.csv',
    'baseline_ckpt': V30 / 'artifacts/mediseg_edgeattnet_refiner_fold2_resume.pt',
    'yolo_ckpt': V30 / 'runs/yolo_fold_2/weights/best.pt',
    'crop_ckpt': V30 / 'runs/v124_croptrain_f2_honest/yolo_croptrain_fold2/weights/best.pt',
    'yolo26_ckpt': V30 / 'runs/v117_yolo26/yolo_yolo26m_fold2/weights/best.pt',
    'v124a_manifest': V30 / 'artifacts/v124a_croptrain_f2_colab_manifest.json',
    'v117_manifest': V30 / 'artifacts/v117_yolo26_colab_manifest.json',
    'kaggle.json': Path('/content/drive/MyDrive/solar_filament_v35/kaggle.json'),
}
missing = {k: str(v) for k, v in inputs.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f'Drive inputs missing: {missing}')
train_json, test_dir, preproc_dir = (inputs['train_json'], inputs['test_dir'],
                                     inputs['preproc_dir'])
folds_csv, baseline_ckpt, yolo_ckpt = (inputs['folds_csv'], inputs['baseline_ckpt'],
                                      inputs['yolo_ckpt'])
crop_ckpt, yolo26_ckpt = inputs['crop_ckpt'], inputs['yolo26_ckpt']
# §2.99 — crop/yolo26 fold-2 重みの trusted-pin は各学習 manifest が正本
# (学習時に記録された weights_sha256 を読み取り、eval cell で byte 照合)
_v124a_man = json.loads(inputs['v124a_manifest'].read_text(encoding='utf-8'))
_v117_man = json.loads(inputs['v117_manifest'].read_text(encoding='utf-8'))
TRUSTED_SHA256 = {
    str(crop_ckpt): _v124a_man['datasets']['croptrain']['weights_sha256'],
    str(yolo26_ckpt): _v117_man['results']['yolo26m']['weights_sha256'],
}
print('trusted pins from training manifests:',
      {Path(k).parent.parent.parent.name: v[:12] for k, v in TRUSTED_SHA256.items()})
if not any(test_dir.iterdir()):
    raise RuntimeError('test_images dir is empty')
EXPECTED_FOLDS_SHA = 'febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e'
actual = hashlib.sha256(folds_csv.read_bytes()).hexdigest()
if actual != EXPECTED_FOLDS_SHA:
    raise RuntimeError(f'fold CSV sha mismatch: {actual} (Drive artifacts/grouped_folds.csv)')
print('folds_csv sha256 OK (true experimental split)')

# [1] Kaggle auth + code/manifest download
os.makedirs('/root/.kaggle', exist_ok=True)
cred = inputs['kaggle.json'].read_bytes()
print('auth: Drive kaggle.json')
try:
    _u = json.loads(cred)['username']
except Exception:
    _u = '?'
print('kaggle user:', _u)
Path('/root/.kaggle/kaggle.json').write_bytes(cred)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'kaggle', 'ultralytics', 'pycocotools'], check=True)

WORK = Path('/content/work'); IN = WORK / 'input'; IN.mkdir(parents=True, exist_ok=True)
def _dl(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('download failed rc=%s: %s\n%s\n%s' % (
            r.returncode, cmd, r.stdout[-800:], r.stderr[-800:]))
def _sha(p):
    d = hashlib.sha256()
    with open(p, 'rb') as h:
        for b in iter(lambda: h.read(1 << 20), b''):
            d.update(b)
    return d.hexdigest()

_dl(['kaggle', 'datasets', 'download', 'shingo257/solar-v73-code',
     '-p', str(IN), '--unzip', '-q'])
_dl(['kaggle', 'datasets', 'download', '-d', 'shingo257/solar-v73-assets',
     '-f', 'v73_reviewed_provenance.json', '-p', str(IN)])

code_dir = next(p.parent for p in IN.rglob('train_refiner_fold.py'))
sys.path.insert(0, str(code_dir))
manifest_path = next(IN.rglob('v73_reviewed_provenance.json'))
from v73_split_provenance import (load_reviewed_manifest, load_fold_mapping,
                                  validate_checkpoint_binding)
manifest = load_reviewed_manifest(manifest_path, EXPECTED_PROVENANCE_SHA, 2)
if manifest['folds_sha256'] != EXPECTED_FOLDS_SHA:
    raise RuntimeError('reviewed split differs')
coco = json.loads(train_json.read_text())
load_fold_mapping(folds_csv, {i['file_name'] for i in coco['images']}, 2,
                  manifest['folds_sha256'])
validate_checkpoint_binding(yolo_ckpt, 'yolo', manifest)
validate_checkpoint_binding(baseline_ckpt, 'baseline', manifest)

# [2] consensus eval variant — embedded copy of repo scratch/run_fold_oof_eval_consensus.py (§2.98 派生・fold-2 honest 再実行)
CONSENSUS_SCRIPT = '"""fold N OOF で複数検出器の提案をクラスタ化し、合意ゲートで採点 (v120)。\n\n`run_fold_oof_eval.py` (v73) からの派生。postprocess は同一:\ndetector fold-N 推論 (imgsz=1280, conf=0.08, iou=0.55, agnostic_nms,\nretina_masks) → arm 別の提案集合 → refiner で 320px crop の\n確率マップ → threshold で shrink → drop_contained(0.90) →\ngreedy_disjoint(min_area=20) → 公式 PQ (annotator x image group 毎、\nIoU>0.5 厳密より大きい)。\n\n差分は提案生成だけ: 複数 --detector の提案 (conf floor=--det-floor) を\nmask-IoU >= --consensus-iou の推移クラスタ (union-find) にまとめ、\narm ごとの合意ルールで放出する:\n  prod        : `yolo` という名の detector の提案のみ conf>=--conf\n  union3      : 全クラスタ (>=1 detector) — max-conf member の mask\n  union3_c26  : max member conf>=--conf のクラスタ (v115 型 union) — 同上\n  cons2of3    : >=2 detector が支持するクラスタ — max-conf member の mask\n  cons2of3_c26: 支持 member のうち conf>=--conf のものが >=2 detector — 同上\n  cons3of3    : 3 detector 全員が支持 — 同上\n\n`yolo` という名の detector は manifest の `yolo` role で binding 検証し、\n他は --trusted-sha256 の byte-pin で受理する (§2.98)。\n\n使い方:\n    python scratch/run_fold_oof_eval_consensus.py \\\n        --train-json train/..._train.json --preproc-dir preprocessed_v30_q96 \\\n        --folds-csv grouped_folds.csv --fold 2 \\\n        --detector yolo=yolo_fold2_best.pt \\\n        --detector crop=v116_crop_best.pt --detector yolo26=v117_best.pt \\\n        --trusted-sha256 <crop_sha> --trusted-sha256 <yolo26_sha> \\\n        --refiner baseline=mediseg_edgeattnet_refiner_fold2_resume.pt \\\n        --out output/v120_eval.json\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\ntry:\n    from scratch.v73_split_provenance import (\n        load_fold_mapping, load_reviewed_manifest, validate_checkpoint_binding,\n        validate_training_corpus_provenance)\nexcept ImportError:\n    from v73_split_provenance import (\n        load_fold_mapping, load_reviewed_manifest, validate_checkpoint_binding,\n        validate_training_corpus_provenance)\ntry:\n    from src.metrics_official import PanopticQuality  # noqa: E402\nexcept ImportError:\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n    from metrics_official import PanopticQuality  # noqa: E402\n\nCROP_SIZE = 320\nCROP_CONTEXT = 1.65\nCONF = 0.26\nMIN_AREA = 20\nDROP_THRESHOLD = 0.90\nEVAL_SIZE = 2048\n\n\ndef load_models():\n    try:\n        from solar_filament.models.mediseg_models import (\n            MediSegEdgeAttUNet, MediSegUNet)\n    except ImportError:\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        from mediseg_models import MediSegEdgeAttUNet, MediSegUNet\n    return MediSegUNet, MediSegEdgeAttUNet\n\n\ndef load_ownership():\n    """canonical apply_ownership を repo / flat dataset 両配置で解決。"""\n    try:\n        from solar_filament.pipeline.ownership import apply_ownership\n    except ImportError:\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n        from ownership import apply_ownership\n    return apply_ownership\n\n\ndef to_rle(annotation, height, width, mask_utils):\n    segmentation = annotation["segmentation"]\n    if isinstance(segmentation, list):\n        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))\n    if isinstance(segmentation.get("counts"), list):\n        return mask_utils.frPyObjects(segmentation, height, width)\n    rle = dict(segmentation)\n    if isinstance(rle["counts"], str):\n        rle["counts"] = rle["counts"].encode("utf-8")\n    return rle\n\n\ndef iou_matrix(truth, pred, mask_utils):\n    if not truth or not pred:\n        return np.zeros((len(truth), len(pred)), np.float64)\n    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)),\n                      dtype=np.float64).T\n\n\ndef mask_square_box(mask, context=CROP_CONTEXT, minimum=96):\n    height, width = mask.shape\n    ys, xs = np.where(mask)\n    if not len(xs):\n        return 0, 0, width, height\n    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2\n    side = int(math.ceil(max(xs.max() - xs.min() + 1,\n                             ys.max() - ys.min() + 1) * context))\n    side = min(max(side, minimum), min(height, width))\n    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))\n    x1, y1 = x0 + side, y0 + side\n    if x0 < 0:\n        x1 -= x0\n        x0 = 0\n    if y0 < 0:\n        y1 -= y0\n        y0 = 0\n    if x1 > width:\n        x0 -= x1 - width\n        x1 = width\n    if y1 > height:\n        y0 -= y1 - height\n        y1 = height\n    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)\n\n\n@torch.no_grad()\ndef refine_probability(refiner, image, proposal):\n    x0, y0, x1, y1 = mask_square_box(proposal)\n    crop = cv2.resize(image[y0:y1, x0:x1], (CROP_SIZE, CROP_SIZE),\n                      interpolation=cv2.INTER_LINEAR)\n    tensor = (torch.from_numpy(crop.transpose(2, 0, 1)[None].copy())\n              .float().cuda() / 255.0)\n    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()\n    return (cv2.resize(probability, (x1 - x0, y1 - y0),\n                       interpolation=cv2.INTER_LINEAR), (x0, y0, x1, y1))\n\n\ndef result_masks(result):\n    if result.masks is None:\n        return [], np.zeros(0, np.float32)\n    height, width = result.orig_shape\n    output = []\n    for mask in result.masks.data.detach().cpu().numpy():\n        if mask.shape != (height, width):\n            mask = cv2.resize(mask, (width, height),\n                              interpolation=cv2.INTER_LINEAR)\n        output.append(mask >= 0.5)\n    confidences = (result.boxes.conf.detach().cpu().numpy().astype(np.float32)\n                   if result.boxes is not None else np.zeros(0, np.float32))\n    return output, confidences\n\n\ndef sha256_of(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train-json", type=Path, required=True)\n    parser.add_argument("--preproc-dir", type=Path, required=True)\n    parser.add_argument("--test-dir", type=Path, required=True)\n    parser.add_argument("--folds-csv", type=Path, required=True)\n    parser.add_argument("--provenance-manifest", type=Path, required=True,\n                        help="reviewed JSON binding YOLO/legacy weights to fold/split")\n    parser.add_argument("--provenance-sha256", required=True,\n                        help="independently reviewed manifest SHA256; never derive at runtime")\n    parser.add_argument("--fold", type=int, required=True, choices=(0, 1, 2))\n    parser.add_argument("--detector", action="append", required=True,\n                        help="name=ckpt_path (複数指定可。`yolo` は manifest binding、"\n                             "他は --trusted-sha256 pin で受理)")\n    parser.add_argument("--refiner", action="append", required=True,\n                        help="name=ckpt_path (複数指定可、同一提案で採点)")\n    parser.add_argument("--trusted-sha256", action="append", default=[],\n                        help="外部検証済み ckpt の sha256 pin (repeatable)。"\n                             "一致バイトは manifest/metadata binding を代替受理")\n    parser.add_argument("--model", type=str, default="edgeatt",\n                        choices=("unet", "edgeatt"))\n    parser.add_argument("--thresholds", type=str, default="0.35,0.40,0.45,0.50")\n    parser.add_argument("--conf", type=float, default=CONF,\n                        help="prod arm / cons2of3_c26 の operating conf")\n    parser.add_argument("--det-floor", type=float, default=0.08,\n                        help="全 detector の提案保持 floor (cluster 入力)")\n    parser.add_argument("--consensus-iou", type=float, default=0.5,\n                        help="cross-detector cluster の mask-IoU 閾値")\n    parser.add_argument("--limit", type=int, default=0)\n    parser.add_argument("--prod-from", type=str, default="yolo",\n                        help="prod arm の提案を供給する detector 名 "\n                             "(default \'yolo\' = manifest binding 必須)。"\n                             "別名にすると `yolo` detector は任意になり、"\n                             "指定 detector は --trusted-sha256 で受理する")\n    parser.add_argument("--eval-files", type=Path, default=None,\n                        help="1 行 1 file_name の評価対象リスト。"\n                             "指定時は oof_files をこの集合に制限 "\n                             "(subset eval 用。全名が fold 対象に存在すること)")\n    parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args()\n    trusted = set(args.trusted_sha256)\n    manifest = load_reviewed_manifest(\n        args.provenance_manifest, args.provenance_sha256, args.fold)\n    if args.limit < 0:\n        raise ValueError("--limit must be non-negative")\n    thresholds = [float(t) for t in args.thresholds.split(",")]\n    arms = []\n    for spec in args.refiner:\n        name, _, path = spec.partition("=")\n        if (not name or name == "yolo" or name in {n for n, _ in arms}\n                or not path or not Path(path).exists()):\n            raise ValueError(f"--refiner name=path が不正: {spec}")\n        arms.append((name, Path(path)))\n\n    detectors = []\n    for spec in args.detector:\n        name, _, path = spec.partition("=")\n        if (not name or not path or not Path(path).exists()\n                or name in {n for n, _ in detectors}):\n            raise ValueError(f"--detector name=path が不正: {spec}")\n        detectors.append((name, Path(path)))\n    det_names = {n for n, _ in detectors}\n    if args.prod_from == "yolo" and "yolo" not in det_names:\n        raise ValueError("--detector yolo=... は必須 (prod arm/binding の基準)")\n    if args.prod_from not in det_names:\n        raise ValueError(\n            f"--prod-from {args.prod_from} に対応する --detector が無い")\n\n    folds_sha256 = hashlib.sha256(args.folds_csv.read_bytes()).hexdigest()\n    with open(args.train_json, encoding="utf-8") as handle:\n        coco = json.load(handle)\n    images = {i["id"]: i for i in coco["images"]}\n    annotations = defaultdict(list)\n    for ann in coco["annotations"]:\n        annotations[ann["image_id"]].append(ann)\n    file_to_ids = defaultdict(list)\n    for image_id, info in images.items():\n        file_to_ids[info["file_name"]].append(image_id)\n    fold_of_file = load_fold_mapping(\n        args.folds_csv, file_to_ids, args.fold, manifest["folds_sha256"])\n    detector_shas = {}\n    for name, path in detectors:\n        digest = sha256_of(path)\n        if name == "yolo":\n            validate_checkpoint_binding(path, "yolo", manifest)\n        elif digest not in trusted:\n            raise ValueError(\n                f"detector {name} bytes match no reviewed binding and is not "\n                f"in --trusted-sha256: {digest[:16]}")\n        detector_shas[name] = digest\n        print(f"detector {name}: sha256={digest[:16]}", flush=True)\n    yolo_sha256 = detector_shas.get("yolo")\n    checked_arms = {}\n    for name, path in arms:\n        checkpoint = torch.load(path, map_location="cpu", weights_only=True)\n        digest = validate_checkpoint_binding(path, name, manifest, checkpoint)\n        provenance = checkpoint.get("init_report", {}).get("corpus_provenance")\n        if str(checkpoint.get("init", "")).startswith("ssl:") or provenance is not None:\n            validate_training_corpus_provenance(provenance, coco, args.test_dir)\n        checked_arms[name] = (checkpoint, digest)\n    # All fold/split and legacy byte bindings pass before any CUDA/model work.\n    oof_files = [n for n in sorted(file_to_ids)\n                 if fold_of_file.get(n) == args.fold]\n    if args.eval_files is not None:\n        allow = {ln.strip() for ln in args.eval_files.read_text().splitlines()\n                 if ln.strip()}\n        missing = sorted(allow - set(oof_files))\n        if missing:\n            raise ValueError(\n                f"--eval-files の {len(missing)} 件が fold-{args.fold} 対象に無い: "\n                f"{missing[:5]}")\n        oof_files = [n for n in oof_files if n in allow]\n    if args.limit:\n        oof_files = oof_files[:args.limit]\n    print({"fold": args.fold, "oof_files": len(oof_files),\n           "arms": [n for n, _ in arms], "folds_sha256": folds_sha256[:16]},\n          flush=True)\n\n    MediSegUNet, MediSegEdgeAttUNet = load_models()\n    apply_ownership = load_ownership()\n    model_cls = MediSegEdgeAttUNet if args.model == "edgeatt" else MediSegUNet\n    from pycocotools import mask as mask_utils\n    from ultralytics import YOLO\n\n    gt_rles = {image_id: [to_rle(a, int(info["height"]), int(info["width"]),\n                                 mask_utils)\n                          for a in annotations[image_id]\n                          if not a.get("iscrowd", 0)]\n               for image_id, info in images.items()}\n\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    refiners = {}\n    for name, path in arms:\n        ckpt, digest = checked_arms.pop(name)\n        model = model_cls(3, 1, False).cuda().eval()\n        model.load_state_dict(ckpt.get("best_model") or ckpt["model"],\n                              strict=True)\n        refiners[name] = {"model": model,\n                          "sha256": digest,\n                          "corpus_provenance": ckpt.get("init_report", {}).get("corpus_provenance"),\n                          "best_dice": ckpt.get("best_dice"),\n                          "init": ckpt.get("init", "pretrained")}\n        print(f"arm {name}: best_dice={refiners[name][\'best_dice\']}",\n              flush=True)\n\n    det_models = {n: YOLO(str(p)) for n, p in detectors}\n    started = time.time()\n\n    # 1) 全 detector の提案を det-floor で保持 (cluster 入力は arm 共通)\n    props_by_file = {}\n    for name in oof_files:\n        image_path = args.preproc_dir / name\n        entries = []\n        for det_name, model in det_models.items():\n            result = model.predict(\n                str(image_path), imgsz=1280, conf=args.det_floor, iou=0.55,\n                agnostic_nms=True, retina_masks=True, verbose=False)[0]\n            masks, confidences = result_masks(result)\n            for m, s in zip(masks, confidences):\n                if s >= args.det_floor and m.sum() >= MIN_AREA:\n                    entries.append((det_name, m, float(s)))\n        props_by_file[name] = entries\n    print(f"proposals done: {time.time() - started:.0f}s", flush=True)\n\n    # 2) cross-detector cluster → arm 別の放出提案集合\n    def _mask_iou(a, b):\n        inter = int(np.logical_and(a, b).sum())\n        return inter / int(np.logical_or(a, b).sum()) if inter else 0.0\n\n    def consensus_clusters(props, iou_thr):\n        n = len(props)\n        parent = list(range(n))\n\n        def find(x):\n            while parent[x] != x:\n                parent[x] = parent[parent[x]]\n                x = parent[x]\n            return x\n        boxes = []\n        for _, m, _ in props:\n            ys, xs = np.nonzero(m)\n            boxes.append((int(xs.min()), int(ys.min()),\n                          int(xs.max()) + 1, int(ys.max()) + 1) if xs.size\n                         else (0, 0, 0, 0))\n        areas = [int(m.sum()) for _, m, _ in props]\n        for i in range(n):\n            ax0, ay0, ax1, ay1 = boxes[i]\n            for j in range(i + 1, n):\n                if props[i][0] == props[j][0]:\n                    continue\n                bx0, by0, bx1, by1 = boxes[j]\n                inter = (max(0, min(ax1, bx1) - max(ax0, bx0))\n                         * max(0, min(ay1, by1) - max(ay0, by0)))\n                # mask IoU <= box_inter / max(mask_area) — 上界未満なら skip\n                if inter < iou_thr * max(areas[i], areas[j]):\n                    continue\n                if _mask_iou(props[i][1], props[j][1]) >= iou_thr:\n                    ri, rj = find(i), find(j)\n                    if ri != rj:\n                        parent[ri] = rj\n        groups = defaultdict(list)\n        for i in range(n):\n            groups[find(i)].append(i)\n        return list(groups.values())\n\n    ARM_NAMES = ("prod", "union3", "union3_c26", "cons2of3",\n                 "cons2of3_c26", "cons3of3")\n    emitted_by_file = {a: {n: [] for n in oof_files} for a in ARM_NAMES}\n    cluster_diag = {"clusters_per_image": [],\n                    "sources_per_cluster": []}\n    for name in oof_files:\n        props = props_by_file[name]\n        emitted_by_file["prod"][name] = [\n            m for d, m, s in props if d == args.prod_from and s >= args.conf]\n        clusters = consensus_clusters(props, args.consensus_iou)\n        cluster_diag["clusters_per_image"].append(len(clusters))\n        for mem in clusters:\n            sources = {props[i][0] for i in mem}\n            strong = {props[i][0] for i in mem\n                      if props[i][2] >= args.conf}\n            mask = props[max(mem, key=lambda i: props[i][2])][1]\n            cluster_diag["sources_per_cluster"].append(len(sources))\n            emitted_by_file["union3"][name].append(mask)\n            if max(props[i][2] for i in mem) >= args.conf:\n                emitted_by_file["union3_c26"][name].append(mask)\n            if len(sources) >= 2:\n                emitted_by_file["cons2of3"][name].append(mask)\n            if len(strong) >= 2:\n                emitted_by_file["cons2of3_c26"][name].append(mask)\n            if len(sources) >= 3:\n                emitted_by_file["cons3of3"][name].append(mask)\n    print(json.dumps({\n        "emitted_per_arm": {a: int(np.mean([len(v[n]) for n in oof_files]))\n                            for a, v in emitted_by_file.items()},\n        "clusters_per_image": float(np.mean(cluster_diag["clusters_per_image"])),\n        "sources_per_cluster": float(np.mean(cluster_diag["sources_per_cluster"]))},\n        indent=1), flush=True)\n\n    # 3) refiner 確率マップを (refiner × proposal arm) 毎に計算 → threshold sweep → PQ\n    report = {"fold": args.fold, "folds_sha256": folds_sha256,\n              "provenance_manifest_sha256": args.provenance_sha256,\n              "detector_sha256": detector_shas,\n              "prod_from": args.prod_from,\n              "yolo_sha256": yolo_sha256,\n              "conf": args.conf, "det_floor": args.det_floor,\n              "consensus_iou": args.consensus_iou, "min_area": MIN_AREA,\n              "drop_threshold": DROP_THRESHOLD, "thresholds": thresholds,\n              "emitted_per_arm": {a: int(np.mean([len(v[n]) for n in oof_files]))\n                                for a, v in emitted_by_file.items()},\n              "oof_files": len(oof_files),\n              "eval_files_path": str(args.eval_files) if args.eval_files else None,\n              "arms": {}}\n    for arm_name, arm in refiners.items():\n        for prop_arm in ARM_NAMES:\n            meters = {t: PanopticQuality() for t in thresholds}\n            key = f"{arm_name}::{prop_arm}"\n            for name in oof_files:\n                image_path = args.preproc_dir / name\n                image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n                proposals = emitted_by_file[prop_arm][name]\n                prob_maps = [refine_probability(arm["model"], image, m)\n                             for m in proposals]\n                for threshold in thresholds:\n                    refined = []\n                    for (probability, box), proposal in zip(\n                            prob_maps, proposals):\n                        x0, y0, x1, y1 = box\n                        mask = np.zeros_like(proposal, dtype=bool)\n                        mask[y0:y1, x0:x1] = probability >= threshold\n                        refined.append(mask & proposal)\n                    predictions = apply_ownership(\n                        refined, np.ones(len(refined)),\n                        rule="drop_contained_then_area_asc",\n                        containment_threshold=DROP_THRESHOLD,\n                        min_area=MIN_AREA, canvas=EVAL_SIZE)\n                    pred_rles = (mask_utils.encode(np.asfortranarray(\n                        np.stack(predictions, axis=-1).astype(np.uint8)))\n                        if predictions else [])\n                    for image_id in file_to_ids[name]:\n                        truth = gt_rles[image_id]\n                        matrix = iou_matrix(truth, pred_rles, mask_utils)\n                        meters[threshold].add_matrix(matrix, len(truth),\n                                                     len(pred_rles))\n            report["arms"][key] = {\n                "refiner": arm_name, "proposal_arm": prop_arm,\n                "ckpt_sha256": arm["sha256"], "init": arm["init"],\n                "corpus_provenance": arm["corpus_provenance"],\n                "best_crop_dice": arm["best_dice"],\n                "emitted": int(np.mean([len(emitted_by_file[prop_arm][n])\n                                        for n in oof_files])),\n                "by_threshold": {str(t): meters[t].summary()\n                                 for t in thresholds},\n            }\n            report["arms"][key]["pq_by_threshold"] = {\n                str(t): round(meters[t].score, 6) for t in thresholds}\n            print({key: report["arms"][key]["pq_by_threshold"]},\n                  flush=True)\n\n    report["minutes"] = round((time.time() - started) / 60, 1)\n    args.out.write_text(json.dumps(report, ensure_ascii=False, indent=2),\n                        encoding="utf-8")\n    print(json.dumps({a: r["pq_by_threshold"]\n                      for a, r in report["arms"].items()},\n                     ensure_ascii=False, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
Path(code_dir / 'run_fold_oof_eval_consensus.py').write_text(CONSENSUS_SCRIPT)
import ast; ast.parse(CONSENSUS_SCRIPT)
print('consensus eval script written:', code_dir / 'run_fold_oof_eval_consensus.py')
for name, p in [('preproc', preproc_dir), ('folds', folds_csv),
                ('baseline', baseline_ckpt), ('yolo', yolo_ckpt),
                ('crop', crop_ckpt), ('yolo26', yolo26_ckpt),
                ('train_json', train_json), ('code', code_dir)]:
    print(name, '->', p)
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
gpu_env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

MIRROR_DIR = Path('/content/drive/MyDrive/solar_filament_v124/work/latest')
def mirror():
    if not os.path.ismount('/content/drive'):
        raise RuntimeError('Drive not mounted')
    MIRROR_DIR.mkdir(parents=True, exist_ok=True)
    digests = {}
    for p in sorted(WORK.rglob('*')):
        if p.is_file() and 'input' not in p.relative_to(WORK).parts:
            dst = MIRROR_DIR / p.relative_to(WORK)
            expected = _sha(p)
            if dst.exists() and _sha(dst) == expected:
                digests[str(p.relative_to(WORK))] = expected; continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst)
            if _sha(dst) != expected:
                raise RuntimeError(f'Drive mirror sha mismatch: {p.name}')
            digests[str(p.relative_to(WORK))] = expected
            print('mirrored', p.name, p.stat().st_size)
    (MIRROR_DIR / 'sha256_manifest.json').write_text(json.dumps(digests, indent=1))
    print('mirror DONE ->', MIRROR_DIR)


In [ ]:
%%time
# [eval] 6 arm — fold-2 OOF で 3 detector の提案クラスタから合意数で放出 (§2.99 補完)
# crop/yolo26 det の byte-pin を実行時に照合 (trusted-sha256 受理の前提)
for path_str, expected in TRUSTED_SHA256.items():
    p = Path(path_str)
    actual = _sha(p)
    if actual != expected:
        raise RuntimeError(f'trusted sha256 mismatch: {p} {actual[:16]} != {expected[:16]}')
    print('trusted sha256 OK:', p, actual[:16])
trusted_args = [x for sha in TRUSTED_SHA256.values() for x in ('--trusted-sha256', sha)]
r = subprocess.run([sys.executable, str(code_dir / 'run_fold_oof_eval_consensus.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--provenance-manifest', str(manifest_path),
                    '--provenance-sha256', EXPECTED_PROVENANCE_SHA,
                    '--detector', f'yolo={yolo_ckpt}',
                    '--detector', f'crop={crop_ckpt}',
                    '--detector', f'yolo26={yolo26_ckpt}',
                    '--model', 'edgeatt',
                    *trusted_args,
                    '--refiner', f'baseline={baseline_ckpt}',
                    '--out', str(WORK / 'v124_eval.json')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-4000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'v124_eval.json').exists():
    raise RuntimeError(f'eval failed rc={r.returncode}')
mirror()


In [ ]:
# 結果サマリ — 第二 fold 判定 (§2.99 補完: cons2of3_c26 の fold-2 clean 複製)
rep = json.load(open(WORK / 'v124_eval.json'))
arms = rep['arms']
pq = lambda a: arms[a]['pq_by_threshold']['0.45']
print(json.dumps({a: r['pq_by_threshold'] for a, r in arms.items()}, indent=2))
print('emitted/image:', rep.get('emitted_per_arm'))
base = pq('baseline::prod')
deltas = {a: round(pq(a) - base, 6) for a in arms if a != 'baseline::prod'}
rep_delta = deltas['baseline::cons2of3_c26']  # §2.99 の複製対象 arm
gate = {
  'prod_pq_0.45': base,
  'sanity_prod_near_v76': abs(base - 0.444238) < 0.001,
  'prod_diff_vs_v76': round(base - 0.444238, 6),
  'delta_by_arm': deltas,
  'cons2of3_c26_delta_fold2': rep_delta,
  'fold0_reference': 'v121_summary.json (fold-0 結果は別途参照)',
  'v120_retracted_delta': 0.01155,  # 漏洩のため撤回済み — 参照のみ
  'decision': ('second_fold_positive' if rep_delta >= 0.005
               else 'second_fold_negative'),
}
print(json.dumps(gate, indent=2))
Path(WORK / 'v124_summary.json').write_text(json.dumps(
    {'eval': rep, 'gate': gate}, indent=2))
mirror()
